# 🤖 Notebook 3: Attention & Transformers
### The Architecture Interpretability Research Studies

---

**Prerequisites:** Notebooks 1 & 2

**What you'll learn:**
1. Why attention was invented and what problem it solves
2. Scaled dot-product attention — step by step
3. Multi-head attention
4. The residual stream — the most important concept for interpretability
5. A complete (small) transformer block
6. Why interpretability researchers focus on attention heads

---

**Why this matters:**
The famous paper 'A Mathematical Framework for Transformer Circuits' — the founding document of mechanistic interpretability — is entirely about understanding attention heads. Every concept in this notebook is something interpretability researchers think about daily.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
print('Libraries loaded!')

---
## Section 1: The Problem Attention Solves

Imagine reading: **"The animal didn't cross the street because **it** was too tired."**

What does 'it' refer to? The animal. Not the street.

For a neural network to figure this out, each word needs to look at *all other words* and decide which are relevant. That mechanism is **attention**.

In [ ]:
sentence = ['The', 'cat', 'sat', 'on', 'the', 'mat']
seq_len = len(sentence)
d_model = 8   # GPT-2 uses 768; we use 8 for clarity

# Random embeddings (a real model learns these from data)
token_embeddings = np.random.randn(seq_len, d_model) * 0.5

print(f'Sentence: {sentence}')
print(f'Sequence length: {seq_len} tokens')
print(f'Embedding dimension: {d_model}')
print(f'Token embeddings shape: {token_embeddings.shape}')
print(f'  -> {seq_len} tokens, each a {d_model}-dim vector')

---
## Section 2: Scaled Dot-Product Attention

Attention works through three learned projections of each token:
- **Query (Q):** What am I looking for?
- **Key (K):** What do I contain?
- **Value (V):** What information do I pass along?

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Steps:
1. Compute how much my Query matches every other token's Key (dot product → score)
2. Normalize with softmax → attention weights (sum to 1)
3. Weighted sum of Values → new, context-aware representation

In [ ]:
d_head = 4  # dimension of Q, K, V projections

W_Q = np.random.randn(d_model, d_head) * 0.5
W_K = np.random.randn(d_model, d_head) * 0.5
W_V = np.random.randn(d_model, d_head) * 0.5

X = token_embeddings
Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print('Q (Queries) shape:', Q.shape, '- each token asks a query')
print('K (Keys) shape:   ', K.shape, '- each token has a key')
print('V (Values) shape: ', V.shape, '- each token has a value')

In [ ]:
# Step 1: Compute attention SCORES
scale = np.sqrt(d_head)
scores = (Q @ K.T) / scale  # shape: (seq_len, seq_len)

print('Attention scores (before softmax):')
print(f'Shape: {scores.shape}  -- one score for every (query, key) pair')
print(np.round(scores, 2))

In [ ]:
# Step 2: Apply CAUSAL MASK (GPT-style: can only look at past tokens)
mask = np.triu(np.ones((seq_len, seq_len)), k=1) * -1e9
scores_masked = scores + mask

# Step 3: Softmax -> attention WEIGHTS
def softmax_2d(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

attn_weights = softmax_2d(scores_masked)

print('Attention weights (rows sum to 1):')
print(np.round(attn_weights, 3))
print(f'Row sums: {np.round(attn_weights.sum(axis=1), 4)}  (all 1.0)')

In [ ]:
# Step 4: Weighted sum of Values
attn_output = attn_weights @ V

print(f'Attention output shape: {attn_output.shape}')

# Visualize the attention pattern
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(attn_weights, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(seq_len))
ax.set_yticks(range(seq_len))
ax.set_xticklabels(sentence, fontsize=11)
ax.set_yticklabels(sentence, fontsize=11)
ax.set_xlabel('Key (token being attended to)', fontsize=10)
ax.set_ylabel('Query (token doing the looking)', fontsize=10)
ax.set_title('Attention Pattern\n(Darker = more attention)', fontsize=12)
plt.colorbar(im, ax=ax)
for i in range(seq_len):
    for j in range(seq_len):
        ax.text(j, i, f'{attn_weights[i,j]:.2f}',
                ha='center', va='center', fontsize=8,
                color='white' if attn_weights[i,j] > 0.5 else 'black')
plt.tight_layout()
plt.show()

print('INTERPRETABILITY INSIGHT: visualizing attention patterns like this')
print('was one of the first interpretability techniques. Researchers found')
print('that specific heads reliably attend to specific linguistic patterns.')

---
## Section 3: The Residual Stream

This is the most important concept in mechanistic interpretability.

In a transformer there is a central 'highway' called the **residual stream**. Every component reads from it and adds its output back:

```
Input embeddings
      |
  Residual stream  <-----------------------+
      |                                    |
  Attention Head reads + writes back ------+
      |                                    |
  MLP reads + writes back -----------------+
      |
  Final stream -> prediction
```

**Why it matters:** The residual stream is the communication channel of the model. Understanding what is *in* the stream at each layer is the central goal of mech interp.

In [ ]:
np.random.seed(7)
tokens = ['The', 'cat', 'sat', 'mat']
n_tokens = 4
d = 8

residual = np.random.randn(n_tokens, d) * 0.5
norms = {'initial': np.linalg.norm(residual, axis=1)}

# Layer 1: Attention adds to the residual stream
attn_contribution = np.random.randn(n_tokens, d) * 0.3
residual = residual + attn_contribution  # RESIDUAL CONNECTION
norms['after attn 1'] = np.linalg.norm(residual, axis=1)

# Layer 1: MLP adds to the residual stream
mlp_contribution = np.random.randn(n_tokens, d) * 0.4
residual = residual + mlp_contribution  # RESIDUAL CONNECTION
norms['after mlp 1'] = np.linalg.norm(residual, axis=1)

# Layer 2: Attention
attn_contribution2 = np.random.randn(n_tokens, d) * 0.25
residual = residual + attn_contribution2
norms['after attn 2'] = np.linalg.norm(residual, axis=1)

print('Residual stream magnitude per token at each stage:')
print('=' * 50)
for stage, norm_vals in norms.items():
    print(f'\n{stage}:')
    for token, norm in zip(tokens, norm_vals):
        bar = chr(9608) * int(norm * 8)
        print(f'  {token:6s}: norm={norm:.3f}  {bar}')

In [ ]:
print('KEY INSIGHT: The residual stream is ADDITIVE')
print('  Final = embeddings + attn_head_1 + mlp_1 + attn_head_2 + ...')
print()
print('This means we can decompose WHAT each component contributed!')
print()
print('In interpretability this is called Direct Logit Attribution (DLA):')
print('  -> Project each component directly onto the vocabulary')
print('  -> See which words each attention head or MLP was voting for')
print('  -> Identify which component was responsible for a prediction')

---
## Section 4: Multi-Head Attention

Real transformers run **multiple attention heads in parallel**. Each head has its own W_Q, W_K, W_V and learns to attend to *different* patterns:
- One head might track pronoun-antecedent relationships
- Another always looks at the previous token
- Another copies names to the output position

Finding and labeling these specialized heads is core circuit analysis work in interpretability.

In [ ]:
def attention_head(X, W_Q, W_K, W_V, causal=True):
    seq_len, d_model = X.shape
    d_head = W_Q.shape[1]
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    scores = (Q @ K.T) / np.sqrt(d_head)
    if causal:
        mask = np.triu(np.ones((seq_len, seq_len)), k=1) * -1e9
        scores = scores + mask
    attn = softmax_2d(scores)
    return attn @ V, attn


def multi_head_attention(X, heads_params, W_O):
    head_outputs = []
    head_attentions = []
    for W_Q, W_K, W_V in heads_params:
        out, attn = attention_head(X, W_Q, W_K, W_V)
        head_outputs.append(out)
        head_attentions.append(attn)
    concatenated = np.concatenate(head_outputs, axis=-1)
    return concatenated @ W_O, head_attentions


n_heads = 3
d_model = 12
d_head = d_model // n_heads  # = 4
seq_len = 6
X = np.random.randn(seq_len, d_model) * 0.5

heads_params = [
    (np.random.randn(d_model, d_head)*0.3,
     np.random.randn(d_model, d_head)*0.3,
     np.random.randn(d_model, d_head)*0.3)
    for _ in range(n_heads)
]
W_O = np.random.randn(n_heads * d_head, d_model) * 0.3

output, attentions = multi_head_attention(X, heads_params, W_O)
print(f'Input shape:  {X.shape}')
print(f'Output shape: {output.shape}  (same -- residual stream preserved)')
print(f'Attention patterns captured: {len(attentions)} (one per head)')

In [ ]:
sentence = ['The', 'cat', 'sat', 'on', 'the', 'mat']
fig, axes = plt.subplots(1, n_heads, figsize=(14, 4))

for h, (ax, attn) in enumerate(zip(axes, attentions)):
    im = ax.imshow(attn, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'Head {h+1}', fontsize=12, fontweight='bold')
    ax.set_xticks(range(seq_len))
    ax.set_yticks(range(seq_len))
    ax.set_xticklabels(sentence, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(sentence, fontsize=9)
    for i in range(seq_len):
        for j in range(seq_len):
            ax.text(j, i, f'{attn[i,j]:.2f}',
                    ha='center', va='center', fontsize=7,
                    color='white' if attn[i,j] > 0.5 else 'black')

plt.suptitle('Multi-Head Attention -- Each head attends differently!', fontsize=11)
plt.tight_layout()
plt.show()

print('Interpretability researchers label each head behavior:')
print('  Head 3.4: previous-token head (always attends one step back)')
print('  Head 5.1: name-mover head (copies names to the output position)')
print('Finding these patterns IS circuit analysis.')

---
## Section 5: The Full Transformer Block

A transformer layer is:
1. **LayerNorm** -> normalize the residual stream
2. **Multi-Head Attention** -> reads and writes to residual stream
3. **Residual add**
4. **LayerNorm** again
5. **MLP** -> reads and writes to residual stream
6. **Residual add**

Stack 12 of these = GPT-2. Stack 96 = GPT-3.

In [ ]:
def layer_norm(x, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    std = x.std(axis=-1, keepdims=True)
    return (x - mean) / (std + eps)


def mlp_layer(x, W1, b1, W2, b2):
    h = x @ W1 + b1
    h_gelu = h * (1 / (1 + np.exp(-1.702 * h)))  # approximate GELU
    return h_gelu @ W2 + b2


class TransformerBlock:
    def __init__(self, d_model, n_heads, d_ff):
        self.d_model = d_model
        self.n_heads = n_heads
        d_head = d_model // n_heads

        self.heads_params = [
            (np.random.randn(d_model, d_head)*0.1,
             np.random.randn(d_model, d_head)*0.1,
             np.random.randn(d_model, d_head)*0.1)
            for _ in range(n_heads)
        ]
        self.W_O = np.random.randn(d_model, d_model) * 0.1
        self.W1 = np.random.randn(d_model, d_ff) * 0.1
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.randn(d_ff, d_model) * 0.1
        self.b2 = np.zeros(d_model)

    def forward(self, x):
        cache = {}
        cache['pre_attn'] = x.copy()

        x_norm = layer_norm(x)
        attn_out, attn_patterns = multi_head_attention(x_norm, self.heads_params, self.W_O)
        cache['attn_output'] = attn_out
        cache['attn_patterns'] = attn_patterns
        x = x + attn_out
        cache['post_attn'] = x.copy()

        x_norm = layer_norm(x)
        mlp_out = mlp_layer(x_norm, self.W1, self.b1, self.W2, self.b2)
        cache['mlp_output'] = mlp_out
        x = x + mlp_out
        cache['post_mlp'] = x.copy()

        return x, cache


np.random.seed(0)
d_model = 16
n_heads = 4
d_ff = 64
seq_len = 6

block = TransformerBlock(d_model, n_heads, d_ff)
x = np.random.randn(seq_len, d_model) * 0.5
output, cache = block.forward(x)

print('Transformer block forward pass complete!')
print(f'  Input shape:  {x.shape}')
print(f'  Output shape: {output.shape}')
print(f'  Cached keys for analysis: {[k for k in cache if not isinstance(cache[k], list)]}')

In [ ]:
sentence = ['The', 'cat', 'sat', 'on', 'the', 'mat']

print('Interpretability Analysis of Transformer Block')
print('=' * 50)

# How much did attention vs MLP contribute?
attn_mag = np.linalg.norm(cache['attn_output'], axis=-1).mean()
mlp_mag  = np.linalg.norm(cache['mlp_output'],  axis=-1).mean()
ratio = attn_mag / (attn_mag + mlp_mag)
print(f'\nAvg attention contribution: {attn_mag:.4f}')
print(f'Avg MLP contribution:       {mlp_mag:.4f}')
print(f'Attention fraction: {ratio:.1%} of total signal')

# How much did the residual stream change per token?
stream_before = cache['pre_attn']
stream_after  = cache['post_mlp']
change = np.linalg.norm(stream_after - stream_before, axis=-1)

print(f'\nResidual stream change per token:')
for token, delta in zip(sentence, change):
    bar = chr(9608) * int(delta * 20)
    print(f'  {token:6s}: {delta:.4f}  {bar}')

print('\nTokens that change MORE are processed more heavily by this layer.')
print('This helps identify which tokens each layer focuses on.')

In [ ]:
# Visualize all 4 attention heads from the block
attn_patterns = cache['attn_patterns']
fig, axes = plt.subplots(1, n_heads, figsize=(16, 4))

for h, ax in enumerate(axes):
    im = ax.imshow(attn_patterns[h], cmap='Purples', vmin=0, vmax=1)
    ax.set_title(f'Head {h+1}', fontsize=11, fontweight='bold')
    ax.set_xticks(range(seq_len))
    ax.set_yticks(range(seq_len))
    ax.set_xticklabels(sentence, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(sentence, fontsize=8)

plt.suptitle('All Attention Heads in the Transformer Block\n'
             'Each head specializes in a different attention pattern', fontsize=11)
plt.tight_layout()
plt.show()

---
## Exercise: Explore Attention Entropy

**Attention entropy** measures how spread out an attention head's weights are.
- **Low entropy** = head attends sharply to one token (specialized behavior)
- **High entropy** = head spreads attention broadly (diffuse / unclear role)

Entropy: $H = -\sum_j w_j \log(w_j + \epsilon)$

Compute the entropy for each head at each query position, then find which head is the most 'focused' (lowest average entropy).

In [ ]:
def attention_entropy(attn_matrix, eps=1e-9):
    """
    attn_matrix: shape (seq_len, seq_len)
    Returns: shape (seq_len,) -- entropy at each query position
    """
    return -np.sum(attn_matrix * np.log(attn_matrix + eps), axis=-1)


print('Attention entropy per head (lower = more focused attention):')
print('=' * 55)
for h, attn in enumerate(attn_patterns):
    entropies = attention_entropy(attn)
    avg_entropy = entropies.mean()
    bar = chr(9608) * int(avg_entropy * 10)
    print(f'  Head {h+1}: avg entropy = {avg_entropy:.3f}  {bar}')

print()
print('YOUR TURN: modify the exercise below.')
print('Plot a bar chart of average entropy per head using plt.bar()')

In [ ]:
# YOUR CODE HERE
# Hint: compute avg_entropies as a list, then call:
# plt.bar(range(n_heads), avg_entropies)
# plt.xlabel('Head'); plt.ylabel('Avg Entropy'); plt.title('...')
# plt.show()


---
## Congratulations -- Notebook 3 Complete!

You have now built from scratch:
- Scaled dot-product attention with causal masking
- Attention pattern visualization
- Multi-head attention
- The residual stream and why it is central to interpretability
- A complete transformer block with cached activations
- Attention entropy analysis

---
## What To Read Next

You are now ready for the real papers:

1. **A Mathematical Framework for Transformer Circuits** (Elhage et al., 2021)
   https://transformer-circuits.pub/2021/framework/index.html

2. **Toy Models of Superposition** (Elhage et al., 2022)
   https://transformer-circuits.pub/2022/toy_model/index.html

3. **Install TransformerLens** for real model analysis:
   pip install transformer_lens
   Then follow Neel Nanda tutorials at https://neelnanda.io

**Notebook 4** (coming soon): Sparse Autoencoders -- the current frontier.